# Người cầm vật sắc nhọn · sharp_02: pre-train COCO → fine-tune vũ khí công khai → fine-tune vũ khí Việt Nam

Notebook này train detector **vật sắc nhọn** theo hướng pre-train rồi fine-tune hai giai đoạn:

1. **Giai đoạn 1 – vũ khí công khai:** từ YOLO11m pre-train COCO, fine-tune trên các bộ công khai
   (dao, mã tấu, kiếm, rìu…) với **một lớp gộp** `sharp_object`. Lớp hiếm (dao phóng lợn chỉ có vài box)
   không còn kéo model yếu đi; nhãn chi tiết 7 lớp trong `class_map.json` vẫn giữ trong dữ liệu để rà soát.
2. **Giai đoạn 2 – vũ khí Việt Nam:** từ `best.pt` giai đoạn 1, **đóng băng backbone** (10 layer đầu),
   lr thấp (1e-4), train trên ảnh VN đã duyệt (lặp ×10) trộn với 25% ảnh công khai để không quên.

Mọi box dẫn tới cùng một cảnh báo **"Phát hiện người cầm vật sắc nhọn"**. Model chỉ tìm **vật**;
việc xác định **người đang cầm** (box vật nằm trong box người đã nới rộng cho tay và cán dài, xác nhận
theo thời gian) làm ở runtime: `extensions/sharp_object/extension.py`.

**Cách chạy:**
1. Vào *Runtime → Change runtime type*, chọn **A100** (L4 cũng chạy được, chậm hơn).
2. Vào *Runtime → Run all*. Đến bước mount Google Drive thì cho phép truy cập.
3. Phiên Colab **tự ngắt** khi chạy xong, hoặc khi gặp lỗi (traceback được ghi ra `ERROR.txt` trên Drive).

**Dữ liệu:**

| Nguồn | Nội dung | Ghi chú |
|---|---|---|
| Zenodo Dangerous Items | knife → `dao_ngan`, machete → `ma_tau` | CC BY-NC 4.0; ảnh chỉ có súng bị bỏ |
| Granada knife (HF) | knife | CC BY 4.0 |
| Open Images | Knife, Kitchen knife, Dagger, Sword, Axe, tối đa 1500 ảnh mỗi lớp | Kèm **ảnh âm khó** không có box: gậy bóng chày, ô, điện thoại, chai |
| COCO | knife | Tối đa 800 ảnh |
| Simuletic | dao trong cảnh CCTV tổng hợp | |
| **Dữ liệu Việt Nam đã duyệt** trên Drive | | Xem bên dưới |

**Dữ liệu Việt Nam**, đọc trong `MyDrive/SENTINEL/datasets/sharp_object_vn/`:
- `vn_weapons_all_260_training_inputs_009.zip`: chỉ dùng phần `supervised/` (32 ảnh, 84 box đã duyệt).
  228 ảnh trong `unlabeled/` **không** được dùng vì chưa có box.
- Thư mục **`labelled/`**: thả vào đây các file export mới sau khi gán nhãn xong, dạng zip hoặc thư mục, theo một trong hai định dạng:
  - *CVAT → Export → YOLO 1.1*;
  - Ultralytics YOLO (`images/` + `labels/` + `data.yaml`).

  Tên lớp phải trùng với `class_map.json`. Ảnh có file nhãn rỗng bị bỏ qua.
  Nếu một ảnh xuất hiện ở nhiều nguồn, bản mới nhất được dùng.
  Nên kèm một `manifest.csv` có cột `image` / `source_group`, để các ảnh cùng một vụ việc nằm chung một tập.
- **Không** dùng `vn_weapons_cvat_batch_00{1,2,3}.zip` hay `vn_weapons_social_cvat_batch_00{4,5}.zip`:
  đó là box gán nháp tự động, chưa được người duyệt.

**Cách chia tập:** theo nhóm nguồn (cùng video, bài báo hoặc vụ việc thì nằm trọn trong một tập).
Ảnh Việt Nam có val/test riêng. Ảnh train Việt Nam được lặp ×6 ở giai đoạn 1 và ×10 ở giai đoạn 2.

**Các bước:**

| Bước | Việc làm | Thời gian ước tính (A100) |
|---|---|---|
| 1 | Tải bộ công khai (chỉ lần đầu; lần sau đọc bản cache trên Drive) | 30–60 phút |
| 2 | Ghép với dữ liệu VN, chia train/val/test | 2–5 phút |
| 3a | **Giai đoạn 1**: fine-tune YOLO11m (COCO) ở 960 px, 1 lớp gộp, tối đa 80 epoch, dừng sớm sau 20 epoch không cải thiện | 1.5–3.5 giờ |
| 3b | **Giai đoạn 2**: từ `best.pt` giai đoạn 1, đóng băng backbone, lr 1e-4, ảnh VN ×10 + 25% ảnh công khai, tối đa 40 epoch | 15–40 phút |
| | Cả hai giai đoạn train trên ổ phiên; checkpoint, `results.csv` và log được chép lên Drive mỗi 5 phút. Chạy lại notebook thì **train tiếp** từ `last.pt` của giai đoạn đang dở | |
| 4 | Đánh giá mAP (không phân lớp) trên toàn bộ test và trên phần VN. Chọn ngưỡng cảnh báo sao cho tỷ lệ báo nhầm trên ảnh âm của val ≤ 3% | ~5 phút |
| 5 | Đóng gói `sharp_object_<RUN_TAG>.zip` (trọng số fp16, spec, kết quả đánh giá) | 1 phút |

**Kết quả trên Drive:** `MyDrive/SENTINEL/artifacts/sharp_object/<RUN_TAG>/`

**Lưu ý:**
- Val/test VN hiện chỉ có vài ảnh, nên điểm trên phần VN chỉ mang tính tham khảo cho tới khi gán thêm nhãn.
- Quyền sử dụng ảnh báo chí, YouTube và Telegram chưa được xác minh. Chỉ dùng cho đồ án / nghiên cứu.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, tarfile, time, traceback, urllib.request, zipfile
from google.colab import drive

RUN_TAG = 'sharp_02'          # đổi tên khi chạy thí nghiệm mới
MODEL = 'yolo11m.pt'          # yolo11s.pt nhanh hơn; yolo11l.pt chính xác hơn nhưng chậm
IMGSZ = 960                   # ảnh lớn giúp thấy lưỡi dao nhỏ
STAGE1_EPOCHS = 80            # giai đoạn 1: vũ khí công khai
STAGE2_EPOCHS = 40            # giai đoạn 2: vũ khí Việt Nam (backbone đóng băng)
FREEZE = 10                   # số layer đầu (backbone) đóng băng ở giai đoạn 2
PUBLIC_POOL_VERSION = 'v1'    # đổi để tải lại bộ công khai
REFRESH_PUBLIC_POOL = False   # True: bỏ cache trên Drive, tải lại bộ công khai
AUTO_DISCONNECT = True        # tự ngắt phiên khi xong hoặc khi lỗi
SOURCE_REVISION = '9e58459'

assert RUN_TAG and all(c.isalnum() or c in '_-' for c in RUN_TAG)
subprocess.run(['nvidia-smi'], check=True)
drive.mount('/content/drive', timeout_ms=180000)
roots = [p for p in (Path('/content/drive/MyDrive'), Path('/content/drive/My Drive')) if p.is_dir()]
assert roots, 'Không thấy MyDrive sau khi mount Google Drive'
SENTINEL = roots[0] / 'SENTINEL'
VN_DIR = SENTINEL / 'datasets' / 'sharp_object_vn'
PUBLIC_CACHE = SENTINEL / 'datasets' / 'sharp_object_public' / f'pool_{PUBLIC_POOL_VERSION}.tar'
OUT = SENTINEL / 'artifacts' / 'sharp_object' / RUN_TAG
OUT.mkdir(parents=True, exist_ok=True)
(VN_DIR / 'labelled').mkdir(parents=True, exist_ok=True)
(OUT / 'drive_write_probe.txt').write_text('write OK', encoding='utf-8')
WORK = Path('/content/sharp_work')
LOCAL_OUT = WORK / 'out'      # train/evaluate/export ghi vào ổ phiên; chỉ chép bản cần giữ lên Drive
LOGS = WORK / 'logs'          # log đầy đủ ở ổ phiên, chép lên Drive định kỳ
SYNC_MINUTES = 5              # chu kỳ chép checkpoint + log lên Drive
WORKERS = 4                   # Colab chỉ có vài nhân CPU; 8 worker + ảnh 960 dễ hết RAM
SRC, POOL, DATA, SCRATCH = WORK / 'src', WORK / 'pool', WORK / 'dataset', WORK / 'scratch'
for path in (WORK, SCRATCH, LOCAL_OUT, LOGS):
    path.mkdir(parents=True, exist_ok=True)
STARTED = time.time()


SYNC_FILES = tuple(f'{stage}train/{name}' for stage in ('stage1/', '')
                   for name in ('weights/last.pt', 'weights/best.pt', 'results.csv', 'args.yaml', 'TRAIN_DONE'))
SYNC_FILES += ('stage2_summary.json', 'evaluation.json')


def sync_to_drive():
    """Chép checkpoint, kết quả và log từ ổ phiên lên Drive (ghi file tạm rồi đổi tên)."""
    import threading
    with globals().setdefault('_SYNC_LOCK', threading.Lock()):
        items = [(LOCAL_OUT / rel, OUT / rel) for rel in SYNC_FILES]
        items += [(p, OUT / p.name) for p in LOGS.glob('*.log')]
        items += [(p, OUT / p.name) for p in LOCAL_OUT.glob('sharp_object_*.zip')]
        for source, target in items:
            if not source.is_file():
                continue
            if target.is_file() and target.stat().st_size == source.stat().st_size \
                    and target.stat().st_mtime >= source.stat().st_mtime:
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            partial = target.with_name(target.name + '.partial')
            shutil.copy2(source, partial)
            partial.replace(target)


def disconnect(reason):
    print(reason, flush=True)
    try:
        sync_to_drive()  # giữ checkpoint/log mới nhất trước khi ngắt
    except Exception as error:
        print('Không chép được lên Drive:', error, flush=True)
    if AUTO_DISCONNECT:
        print('Phiên Colab sẽ tự ngắt sau 20 giây (đặt AUTO_DISCONNECT = False để giữ lại).', flush=True)
        time.sleep(20)  # để Drive kịp ghi xong
        drive.flush_and_unmount()
        from google.colab import runtime
        runtime.unassign()


def on_error(shell, etype, evalue, tb, tb_offset=None):
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    text = ''.join(traceback.format_exception(etype, evalue, tb))
    (OUT / 'ERROR.txt').write_text(time.strftime('%Y-%m-%d %H:%M:%S ') + text, encoding='utf-8')
    disconnect(f'LỖI, xem {OUT / "ERROR.txt"} và log trên Drive.')


get_ipython().set_custom_exc((Exception,), on_error)  # lỗi ở bất kỳ cell nào -> ghi Drive + tự ngắt
(OUT / 'ERROR.txt').unlink(missing_ok=True)
print('Kết quả sẽ lưu tại:', OUT)
print('Dữ liệu VN đọc từ:', VN_DIR)

In [ ]:
# Mã nguồn được ghim theo commit để kết quả tái lập được.
marker = SRC / '.revision'
if not marker.is_file() or marker.read_text().strip() != SOURCE_REVISION:
    shutil.rmtree(SRC, ignore_errors=True)
    archive = WORK / 'source.zip'
    url = 'https://codeload.github.com/TranDuon/anomaly-behavior-detector-system/zip/' + SOURCE_REVISION
    with urllib.request.urlopen(url, timeout=300) as response, archive.open('wb') as output:
        shutil.copyfileobj(response, output)
    with zipfile.ZipFile(archive) as package:
        top = package.namelist()[0].split('/')[0]
        package.extractall(WORK)
    (WORK / top).rename(SRC)
    marker.write_text(SOURCE_REVISION)
print('Mã nguồn:', SOURCE_REVISION)


def is_progress(line):
    """Dòng thanh tiến trình theo batch của Ultralytics/tqdm (chỉ ghi log, không in ra notebook)."""
    return '%|' in line or ('it/s]' in line) or ('s/it]' in line)


def run(args, log_name, cwd=SRC):
    """Chạy lệnh; log đầy đủ ghi ở ổ phiên (chép lên Drive định kỳ), notebook chỉ hiện dòng tóm tắt."""
    started = time.time()
    env = {**os.environ, 'TQDM_MININTERVAL': '30'}
    with (LOGS / log_name).open('a', encoding='utf-8') as log:
        log.write(f'\n$ {" ".join(map(str, args))}\n')
        process = subprocess.Popen(list(map(str, args)), cwd=cwd, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        for line in process.stdout:
            log.write(line)
            if not is_progress(line):
                print(line, end='')
        code = process.wait()
    print(f'--- {log_name}: exit {code}, {(time.time() - started) / 60:.1f} phút')
    if code:
        raise RuntimeError(f'{log_name} lỗi (exit {code}), xem {OUT / log_name}')


def py(module, *args, log):
    run([sys.executable, '-u', '-m', module, *args], log)

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.183', 'hf_xet'], 'install.log')
import torch, ultralytics
assert torch.cuda.is_available(), 'Chọn GPU (A100/L4) trong Runtime → Change runtime type'
GPU_MEMORY = torch.cuda.get_device_properties(0).total_memory
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__, '| ultralytics', ultralytics.__version__)
print('Ổ đĩa trống: %.0f GB' % (shutil.disk_usage('/content').free / 1e9))

## Bước 1 · Bộ dữ liệu công khai (đã có box)

Chỉ chạy ở lần đầu. Pool được đóng gói thành `pool_<version>.tar` trên Drive, lần sau chỉ cần giải nén.
Nguồn nào lỗi mạng thì bỏ qua và in cảnh báo, các nguồn còn lại vẫn chạy.

In [ ]:
if PUBLIC_CACHE.is_file() and not REFRESH_PUBLIC_POOL:
    print('Dùng pool công khai đã cache:', PUBLIC_CACHE, '(%.2f GB)' % (PUBLIC_CACHE.stat().st_size / 1e9))
    shutil.rmtree(POOL, ignore_errors=True)
    with tarfile.open(PUBLIC_CACHE) as bundle:
        bundle.extractall(WORK)
else:
    failed = []
    for source, extra in (('zenodo', []), ('granada', []), ('simuletic', []),
                          ('coco', ['--coco-limit', 800]),
                          ('openimages', ['--oi-per-class', 1500, '--oi-negatives-per-class', 300])):
        try:
            py('extensions.sharp_object.collect', 'labeled', '--root', POOL, '--work', SCRATCH,
               '--sources', source, *extra, log='collect.log')
        except RuntimeError as error:
            failed.append(source)
            print('CẢNH BÁO: bỏ qua nguồn', source, '-', error)
        shutil.rmtree(SCRATCH, ignore_errors=True)
        SCRATCH.mkdir(parents=True, exist_ok=True)
    assert (POOL / 'manifest.csv').is_file(), 'Không tải được nguồn công khai nào'
    PUBLIC_CACHE.parent.mkdir(parents=True, exist_ok=True)
    partial = PUBLIC_CACHE.with_suffix('.partial')
    with tarfile.open(partial, 'w') as bundle:
        bundle.add(POOL, arcname='pool')
    partial.replace(PUBLIC_CACHE)
    print('Đã cache pool lên Drive:', PUBLIC_CACHE, '| nguồn lỗi:', failed or 'không')
print(json.dumps(json.loads((POOL / 'stats.json').read_text()), indent=1, ensure_ascii=False))

## Bước 2 · Ghép dữ liệu VN đã duyệt và chia tập

Thứ tự ưu tiên đầu vào VN (cái sau thay cái trước khi trùng ảnh):
1. `vn_weapons_all_260_training_inputs_009.zip`
2. mọi zip và thư mục trong `labelled/`, sắp theo thời gian sửa (mới nhất cuối cùng)

In [ ]:
vn_inputs = [VN_DIR / 'vn_weapons_all_260_training_inputs_009.zip']
vn_inputs = [p for p in vn_inputs if p.exists()]
vn_inputs += sorted((p for p in (VN_DIR / 'labelled').iterdir() if p.suffix.lower() == '.zip' or p.is_dir()),
                    key=lambda p: p.stat().st_mtime)
print('Đầu vào VN:')
for path in vn_inputs:
    print('  ', path.relative_to(VN_DIR))
local_inputs = []
for index, path in enumerate(vn_inputs):  # chép về ổ phiên để đọc nhanh
    target = WORK / 'vn_inputs' / f'{index:02d}_{path.name}'
    target.parent.mkdir(parents=True, exist_ok=True)
    if path.is_dir():
        shutil.copytree(path, target, dirs_exist_ok=True)
    else:
        shutil.copy(path, target)
    local_inputs.append(target)
py('extensions.sharp_object.build_dataset', '--pool', POOL, '--vn', *local_inputs, '--out', DATA,
   '--vn-repeat', 6, log='build_dataset.log')
shutil.copy(DATA / 'summary.json', OUT / 'dataset_summary.json')
shutil.copy(DATA / 'manifest.csv', OUT / 'dataset_manifest.csv')
summary = json.loads((DATA / 'summary.json').read_text())
for split in ('train', 'val', 'test'):
    print(split, '| ảnh:', summary['images'][split], '| VN:', summary[split]['vn_images'],
          '| ảnh âm:', summary[split]['negatives'], '| box theo lớp:', summary[split]['boxes_by_class'])
for warning in summary['warnings']:
    print('CẢNH BÁO:', warning)

## Bước 3 · Fine-tune hai giai đoạn

**Giai đoạn 1 – vũ khí công khai** (`stage1/train/` trên Drive)
- Khởi đầu từ YOLO11m pre-train COCO; **1 lớp gộp** (`--single-cls`); AdamW, lr0 = 1e-3, cosine LR;
  tắt mosaic ở 15 epoch cuối; tối đa `STAGE1_EPOCHS` epoch, dừng sớm sau 20 epoch không cải thiện.

**Giai đoạn 2 – vũ khí Việt Nam** (`train/` trên Drive, là kết quả cuối)
- Khởi đầu từ `stage1/train/weights/best.pt`; đóng băng `FREEZE` layer đầu (backbone), lr0 = 1e-4,
  warmup 1 epoch, tối đa `STAGE2_EPOCHS` epoch, dừng sớm sau 15 epoch không cải thiện.
- Danh sách train: ảnh VN đã duyệt lặp ×10 + 25% ảnh train công khai (giữ kiến thức giai đoạn 1).
  Thống kê ghi ở `stage2_summary.json`. Không có ảnh VN nào thì giữ nguyên kết quả giai đoạn 1.

**Chung cho cả hai giai đoạn**
- Train ghi vào ổ phiên `/content` (nhanh, không treo Drive). Một luồng nền chép `last.pt`, `best.pt`, `results.csv` và log lên Drive mỗi `SYNC_MINUTES` phút, và một lần nữa khi xong, lỗi hoặc tự ngắt.
- Dùng `WORKERS = 4` dataloader (8 worker với ảnh 960 px dễ làm hết RAM phiên). Notebook chỉ in dòng tóm tắt; thanh tiến trình theo batch chỉ ghi vào log.
- Nếu phiên bị ngắt giữa chừng, chỉ cần *Run all* lại: dữ liệu được dựng lại y hệt, bản trên Drive được chép về và train tiếp từ `last.pt` của giai đoạn đang dở (mất tối đa khoảng 5 phút train cuối).


In [ ]:
# Train vào ổ phiên (/content), không ghi thẳng checkpoint 200 MB lên Drive mỗi epoch.
# Một luồng nền chép last.pt / best.pt / results.csv / log lên Drive mỗi SYNC_MINUTES phút.
# Bị ngắt phiên: chạy lại notebook, bản trên Drive được chép về và train tiếp từ last.pt.
import threading, torch

STAGE1 = LOCAL_OUT / 'stage1'
for drive_run, local_run in ((OUT / 'stage1' / 'train', STAGE1 / 'train'), (OUT / 'train', LOCAL_OUT / 'train')):
    if (drive_run / 'weights' / 'last.pt').is_file() and not (local_run / 'weights' / 'last.pt').is_file():
        print('Chép run dở từ Drive về ổ phiên:', drive_run)
        shutil.copytree(drive_run, local_run, dirs_exist_ok=True)
    last = local_run / 'weights' / 'last.pt'
    if last.is_file() and not (local_run / 'TRAIN_DONE').is_file():
        # Checkpoint ghi đường dẫn lưu cũ; sửa để train tiếp ghi đúng vào ổ phiên của phiên này.
        ckpt = torch.load(last, map_location='cpu', weights_only=False)
        train_args = ckpt.get('train_args') or {}
        train_args.update(project=str(local_run.parent), name='train', save_dir=str(local_run), workers=WORKERS,
                          exist_ok=True)
        ckpt['train_args'] = train_args
        torch.save(ckpt, last)
        print(local_run, '-> train tiếp từ epoch', ckpt.get('epoch', -1) + 1)

stop_sync = threading.Event()


def sync_loop():
    while not stop_sync.wait(SYNC_MINUTES * 60):
        try:
            sync_to_drive()
        except Exception as error:  # Drive chậm/lỗi tạm thời không được làm dừng train
            print('Chép lên Drive lỗi (sẽ thử lại):', error, flush=True)


syncer = threading.Thread(target=sync_loop, daemon=True)
syncer.start()
batch = 16 if GPU_MEMORY > 30e9 else 8
common = ['--data', DATA, '--model', MODEL, '--imgsz', IMGSZ, '--batch', batch, '--workers', WORKERS, '--single-cls']
try:
    print('=== Giai đoạn 1: vũ khí công khai (1 lớp gộp) ===')
    py('extensions.sharp_object.train_sharp_object', '--stage', 'train', '--output', STAGE1, *common,
       '--epochs', STAGE1_EPOCHS, '--patience', 20, log='train_stage1.log')
    sync_to_drive()
    print('=== Giai đoạn 2: fine-tune vũ khí Việt Nam (backbone đóng băng) ===')
    py('extensions.sharp_object.train_sharp_object', '--stage', 'finetune_vn', '--output', LOCAL_OUT, *common,
       '--init', STAGE1 / 'train' / 'weights' / 'best.pt', '--epochs', STAGE2_EPOCHS, '--patience', 15,
       '--lr0', 1e-4, '--freeze', FREEZE, '--close-mosaic', 10, '--vn-repeat', 10, '--replay', 0.25,
       log='train_stage2.log')
finally:
    stop_sync.set()
    sync_to_drive()
print('Checkpoint trên Drive:', OUT / 'stage1' / 'train' / 'weights', 'và', OUT / 'train' / 'weights')


## Bước 4 · Đánh giá và chọn ngưỡng cảnh báo

- **Box mAP** không phân lớp (model 1 lớp gộp; mọi loại vũ khí đều dẫn tới cùng một cảnh báo), trên toàn bộ val/test và trên phần VN.
- **Cảnh báo theo ảnh**: một ảnh bị coi là có cảnh báo khi có box ≥ ngưỡng.
  Ngưỡng là giá trị thấp nhất mà tỷ lệ báo nhầm trên ảnh âm của val ≤ 3%.
  Với ngưỡng đó, đo recall và tỷ lệ báo nhầm trên test.

In [ ]:
py('extensions.sharp_object.train_sharp_object', '--stage', 'evaluate', '--data', DATA, '--output', LOCAL_OUT,
   '--imgsz', IMGSZ, '--max-false-alarm', 0.03, log='evaluate.log')
sync_to_drive()


## Bước 5 · Đóng gói

In [ ]:
py('extensions.sharp_object.train_sharp_object', '--stage', 'export', '--data', DATA, '--output', LOCAL_OUT,
   '--imgsz', IMGSZ, '--run-tag', RUN_TAG, log='export.log')
sync_to_drive()  # zip + evaluation.json + log lên Drive
report = json.loads((OUT / 'evaluation.json').read_text())
fmt = lambda d: {k: (round(v, 3) if isinstance(v, float) else v) for k, v in d.items() if k != 'per_class'}
for key in ('box_all_test_agnostic', 'box_all_test', 'box_vn_test_agnostic', 'box_vn_val_agnostic'):
    print(f'{key:24s}', fmt(report[key]))
print('\nAP50 test theo lớp:', {k: round(v['ap50'], 3) for k, v in report['box_all_test'].get('per_class', {}).items()})
print('\nNgưỡng cảnh báo:', report['alert_threshold'], '-', report['threshold_rule'])
for key in ('alert_val', 'alert_test', 'alert_vn_test'):
    print(f'{key:16s}', fmt(report[key]))
print('\nFile trên Drive:')
for path in sorted(OUT.rglob('*')):
    if path.is_file() and path.suffix in {'.zip', '.json', '.pt', '.log', '.csv', '.txt'}:
        print(f'  {str(path.relative_to(OUT)):48s} {path.stat().st_size / 1e6:8.2f} MB')
(OUT / 'DONE.txt').write_text(f'xong {time.strftime("%Y-%m-%d %H:%M:%S")}, {(time.time() - STARTED) / 3600:.1f} giờ',
                              encoding='utf-8')
print('\nTổng thời gian: %.1f giờ' % ((time.time() - STARTED) / 3600))
print('Bước tiếp theo: tải sharp_object_' + RUN_TAG + '.zip về máy để đo trên video thật và tích hợp runtime.')

## Tự ngắt phiên

Cell cuối ngắt phiên Colab để không tốn compute unit, vì kết quả đã nằm trên Drive.

In [ ]:
disconnect('XONG. Kết quả ở ' + str(OUT))